In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

2 channel Terms of Service accepted
Channels:
 - pytorch
 - nvidia
 - defaults
Platform: linux-64
Solving environment: failed

LibMambaUnsatisfiableError: Encountered problems while solving:
  - nothing provides python_abi 3.7.* *_cp37m needed by torchvision-0.13.0-py37_cu116

Could not solve for environment specs
The following packages are incompatible
├─ cuda-version =12.4 * is requested and can be installed;
├─ pin on python =3.13 * is installable and it requires
│  └─ python =3.13 *, which can be installed;
├─ rapids =26.2 * is installable and it requires
│  ├─ custreamz =26.2 *, which requires
│  │  └─ python-confluent-kafka >=2.8.0,<2.9.0 *, which requires
│  │     └─ librdkafka >=2.8.0 *, which requires
│  │        └─ lz4-c >=1.10.0,<1.11.0a0 *, which can be installed;
│  └─ cuxfilter =26.2 *, which requires
│     └─ geopandas >=0.11.0 * with the potential options
│        ├─ geopandas 0.12.2 would require
│        │  └─ python >=3.8,<3.9.0a0 *, which conflicts with any installa

In [9]:
import cudf
import cupy as cp
import torch

In [24]:
mapping_df = cudf.read_feather("/workspace/data/450_probe_to_feature_mapping.feather")

/opt/conda/envs/rapids-env/lib/python3.13/site-packages/cudf/io/feather.py:16: UserWarning: Using CPU via PyArrow to read feather dataset, this may be GPU accelerated in the future
  warnings.warn(


In [11]:
m_values_df = cudf.read_feather("/workspace/data/GSE111629_data_test.feather")

In [25]:
mapping_df = mapping_df.set_index("Probe_ID")

In [22]:
m_values_df.head()

,Sentrix_ID,Sample_Name,Basename,title,geo_accession,status,submission_date,last_update_date,type,channel_count,...,cg19324023,cg09635994,cg19004771,cg20569369,cg26034629,cg25232725,cg05615487,cg22122449,cg08423507,cg19565306
0,3999979001_R01C01,GSM3035401,GSE111629_RAW/GSM3035401_3999979001_R01C01,genomic DNA from 3999979001_R01C01,GSM3035401,Public on Mar 10 2018,Mar 09 2018,Mar 10 2018,genomic,1,...,3.354123,2.004086,1.929260,2.892161,4.913450,2.807751,4.782970,4.457486,5.246001,-6.148653
1,3999979001_R01C02,GSM3035402,GSE111629_RAW/GSM3035402_3999979001_R01C02,genomic DNA from 3999979001_R01C02,GSM3035402,Public on Mar 10 2018,Mar 09 2018,Mar 10 2018,genomic,1,...,4.178481,1.938704,2.545114,2.160673,4.978455,2.912098,4.853534,4.736333,5.141731,-6.166355
2,3999979001_R02C01,GSM3035403,GSE111629_RAW/GSM3035403_3999979001_R02C01,genomic DNA from 3999979001_R02C01,GSM3035403,Public on Mar 10 2018,Mar 09 2018,Mar 10 2018,genomic,1,...,3.367879,1.731055,1.931527,2.390308,5.219790,2.573055,4.933959,5.374524,5.142147,-6.167470
3,3999979001_R02C02,GSM3035404,GSE111629_RAW/GSM3035404_3999979001_R02C02,genomic DNA from 3999979001_R02C02,GSM3035404,Public on Mar 10 2018,Mar 09 2018,Mar 10 2018,genomic,1,...,2.990683,2.210517,1.962918,2.375698,4.828473,3.111890,4.963863,3.586690,5.095090,-6.178808
4,3999979001_R03C01,GSM3035405,GSE111629_RAW/GSM3035405_3999979001_R03C01,genomic DNA from 3999979001_R03C01,GSM3035405,Public on Mar 10 2018,Mar 09 2018,Mar 10 2018,genomic,1,...,3.262536,1.902270,1.696990,2.303311,5.145214,2.800162,4.069010,4.884013,5.142816,-6.166770


`m_valhes_df` has probes as cols and samples as rows. The mapping to genomic regions needs it the other way around.
Before transposing, the samples need to be set as the dataframe index.

In [26]:
m_values_df = m_values_df.set_index("Sample_Name")

In [29]:
all_cols = m_values_df.columns.to_list()

In [49]:
probe_cols = [c for c in all_cols if c.startswith('cg') or (c.startswith('ch') and not c.startswith('cha'))]

In [50]:
probe_set = set(probe_cols)
pheno_cols = [c for c in all_cols if c not in probe_set]

In [51]:
pheno_df = m_values_df[pheno_cols]
m_values_wide = m_values_df[probe_cols]

In [54]:
m_values_df = m_values_wide.T
m_values_df.index.name = "Probe_ID"

In [55]:
merged_df = m_values_df.join(mapping_df, how="inner")

In [56]:
grouped = merged_df.groupby("Feature_Name")

In [57]:
mean_df = grouped.mean()
var_df = grouped.var()

In [58]:
var_df = var_df.fillna(0.0)

In [59]:
ordered_regions = mean_df.index.to_pandas().tolist()

In [62]:
import cudf
import torch

patient_graphs = {}
sample_ids = mean_df.columns.tolist()

for sample in sample_ids:
    patient_means = mean_df[sample].values
    patient_vars = var_df[sample].values
    
    means_dlpack = patient_means.toDlpack()
    vars_dlpack = patient_vars.toDlpack()
    
    # PyTorch natively understands DLPack and bypasses the ATen_cuda initialization check
    tensor_means = torch.utils.dlpack.from_dlpack(means_dlpack).to(torch.float32)
    tensor_vars = torch.utils.dlpack.from_dlpack(vars_dlpack).to(torch.float32)
    
    # 4. Stack horizontally [N, 2]
    X = torch.stack((tensor_means, tensor_vars), dim=1)
    
    patient_graphs[sample] = X

print(f"Shape of X tensor: {patient_graphs[sample_ids[0]].shape}")

/tmp/ipykernel_1702/1638117370.py:11: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  means_dlpack = patient_means.toDlpack()
/tmp/ipykernel_1702/1638117370.py:12: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  vars_dlpack = patient_vars.toDlpack()


AssertionError: Torch not compiled with CUDA enabled

In [61]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("FATAL: PyTorch cannot see the GPU.")

PyTorch Version: 2.13.0
CUDA Available: False
FATAL: PyTorch cannot see the GPU.


In [63]:
%conda remove pytorch torchvision torchaudio -y

2 channel Terms of Service accepted

PackagesNotFoundInPrefixError: The following packages are missing from the target environment:
  prefix: /opt/conda/envs/rapids-env

  - torchaudio
  - torchvision



Note: you may need to restart the kernel to use updated packages.


In [64]:
%conda clean --all -y

Will remove 27 (263.8 MB) tarball(s).
Will remove 1 index cache(s).
Will remove 4 (218 KB) package(s).
There are no tempfile(s) to remove.
There are no logfile(s) to remove.

Note: you may need to restart the kernel to use updated packages.
